In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    if filenames:  # check if directory has files
        print(os.path.join(dirname, filenames[0]))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/utd-mhad-1/Skeleton/Skeleton/a17_s1_t2_skeleton.mat
/kaggle/input/utd-mhad-1/Inertial/Inertial/a23_s1_t1_inertial.mat
/kaggle/input/utd-mhad-1/Depth/Depth/a8_s4_t2_depth.mat


In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os
import glob
import cv2
from scipy.io import loadmat

# ================== Multi-Modal Dataset Class ==================
class UTDMHADMultiModalDataset(Dataset):
    """
    Multi-modal dataset for UTD-MHAD (Skeleton + Inertial + Depth)
    """
    def __init__(self, data_dir, split='train', test_size=0.2, random_state=42, 
                 use_skeleton=True, use_inertial=True, use_depth=True):
        """
        Args:
            data_dir: Root directory containing Skeleton/, Inertial/, Depth/ folders
            split: 'train' or 'test'
            test_size: Proportion of data for testing
            random_state: Random seed
            use_skeleton: Whether to use skeleton modality
            use_inertial: Whether to use inertial modality
            use_depth: Whether to use depth modality
        """
        self.data_dir = data_dir
        self.split = split
        self.use_skeleton = use_skeleton
        self.use_inertial = use_inertial
        self.use_depth = use_depth
        
        self.skeleton_dir = os.path.join(data_dir, 'Skeleton/Skeleton')
        self.inertial_dir = os.path.join(data_dir, 'Inertial/Inertial')
        self.depth_dir = os.path.join(data_dir, 'Depth/Depth')
        
        # Load file paths and labels
        self.samples = []
        self.labels = []
        
        # Get all skeleton files as reference (.mat format)
        skeleton_files = sorted(glob.glob(os.path.join(self.skeleton_dir, '*.mat')))
        
        for skeleton_file in skeleton_files:
            filename = os.path.basename(skeleton_file)
            # Parse: a{action}_s{subject}_t{trial}_skeleton.mat
            parts = filename.replace('.mat', '').split('_')
            action = int(parts[0][1:]) - 1  # 0-indexed
            subject = int(parts[1][1:])
            trial = int(parts[2][1:])
            
            # Construct paths for all modalities
            base_name = f"a{action+1}_s{subject}_t{trial}"
            
            sample_dict = {
                'action': action,
                'subject': subject,
                'trial': trial
            }
            
            if use_skeleton:
                sample_dict['skeleton_path'] = skeleton_file
            
            if use_inertial:
                inertial_file = os.path.join(self.inertial_dir, f"{base_name}_inertial.mat")
                if os.path.exists(inertial_file):
                    sample_dict['inertial_path'] = inertial_file
                else:
                    continue  # Skip if inertial data missing
            
            if use_depth:
                # Depth data is also in .mat format
                depth_file = os.path.join(self.depth_dir, f"{base_name}_depth.mat")
                if os.path.exists(depth_file):
                    sample_dict['depth_path'] = depth_file
                else:
                    continue  # Skip if depth data missing
            
            self.samples.append(sample_dict)
            self.labels.append(action)
        
        self.labels = np.array(self.labels)
        
        # Train-test split
        indices = np.arange(len(self.samples))
        train_idx, test_idx = train_test_split(
            indices, test_size=test_size, random_state=random_state, stratify=self.labels
        )
        
        if split == 'train':
            self.samples = [self.samples[i] for i in train_idx]
            self.labels = self.labels[train_idx]
        else:
            self.samples = [self.samples[i] for i in test_idx]
            self.labels = self.labels[test_idx]
        
        print(f"{split.upper()} set: {len(self.samples)} samples")
        print(f"Modalities: Skeleton={use_skeleton}, Inertial={use_inertial}, Depth={use_depth}")
    
    def load_skeleton_data(self, file_path, target_frames=50):
        """Load and preprocess skeleton data from .mat file"""
        try:
            mat_data = loadmat(file_path)
            
            # UTD-MHAD skeleton data key (usually 'd_skel')
            # Shape can be (num_frames, 60) or flattened
            if 'd_skel' in mat_data:
                skeleton_data = mat_data['d_skel']
            elif 'skeleton' in mat_data:
                skeleton_data = mat_data['skeleton']
            else:
                # Try to find the right key
                possible_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                if possible_keys:
                    skeleton_data = mat_data[possible_keys[0]]
                else:
                    raise ValueError("Cannot find skeleton data in .mat file")
            
            # Flatten if needed and calculate number of frames
            skeleton_data = skeleton_data.flatten()
            
            # Each frame has 20 joints * 3 coordinates = 60 values
            total_values = skeleton_data.shape[0]
            num_frames = total_values // 60
            
            if num_frames == 0:
                raise ValueError(f"Invalid skeleton data size: {total_values}")
            
            # Reshape to (num_frames, 20 joints, 3 coordinates)
            skeleton_seq = skeleton_data[:num_frames * 60].reshape(num_frames, 20, 3)
            
            # Normalize skeleton data (center and scale)
            skeleton_seq = self.normalize_skeleton(skeleton_seq)
            
            # Pad or truncate to target_frames
            if skeleton_seq.shape[0] < target_frames:
                padding = np.zeros((target_frames - skeleton_seq.shape[0], 20, 3))
                skeleton_seq = np.concatenate([skeleton_seq, padding], axis=0)
            else:
                # Use uniform sampling for truncation to preserve temporal information
                indices = np.linspace(0, skeleton_seq.shape[0] - 1, target_frames).astype(int)
                skeleton_seq = skeleton_seq[indices]
            
            return skeleton_seq.astype(np.float32)
        except Exception as e:
            print(f"Error loading skeleton {file_path}: {e}")
            return np.zeros((target_frames, 20, 3), dtype=np.float32)
    
    def normalize_skeleton(self, skeleton):
        """Normalize skeleton by centering on torso and scaling"""
        # Use joint 1 (spine base) as reference
        if skeleton.shape[0] > 0:
            center = skeleton[:, 1:2, :]  # Spine base
            skeleton = skeleton - center
            
            # Scale by max distance
            max_dist = np.max(np.abs(skeleton)) + 1e-6
            skeleton = skeleton / max_dist
        
        return skeleton
    
    def load_inertial_data(self, file_path, target_frames=50):
        """Load and preprocess inertial data from .mat file"""
        try:
            mat_data = loadmat(file_path)
            
            # Inertial data key (usually 'd_iner')
            # Shape: (num_samples, 6) - [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
            if 'd_iner' in mat_data:
                inertial_data = mat_data['d_iner']
            elif 'inertial' in mat_data:
                inertial_data = mat_data['inertial']
            else:
                # Try to find the right key
                possible_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                if possible_keys:
                    inertial_data = mat_data[possible_keys[0]]
                else:
                    raise ValueError("Cannot find inertial data in .mat file")
            
            # Ensure correct shape
            if inertial_data.shape[1] != 6:
                # Sometimes data might be transposed
                if inertial_data.shape[0] == 6:
                    inertial_data = inertial_data.T
            
            # Resample to target frames
            if inertial_data.shape[0] != target_frames:
                indices = np.linspace(0, inertial_data.shape[0] - 1, target_frames).astype(int)
                inertial_data = inertial_data[indices]
            
            # Normalize
            inertial_data = (inertial_data - np.mean(inertial_data, axis=0)) / (np.std(inertial_data, axis=0) + 1e-6)
            
            return inertial_data.astype(np.float32)
        except Exception as e:
            print(f"Error loading inertial {file_path}: {e}")
            return np.zeros((target_frames, 6), dtype=np.float32)
    
    def load_depth_data(self, file_path, target_frames=50, target_size=(64, 64)):
        """Load and preprocess depth data from .mat file"""
        try:
            mat_data = loadmat(file_path)
            
            # Depth data key (usually 'd_depth')
            # Shape can vary: (height, width, num_frames) or (num_frames, height, width)
            if 'd_depth' in mat_data:
                depth_data = mat_data['d_depth']
            elif 'depth' in mat_data:
                depth_data = mat_data['depth']
            else:
                # Try to find the right key
                possible_keys = [k for k in mat_data.keys() if not k.startswith('__')]
                if possible_keys:
                    depth_data = mat_data[possible_keys[0]]
                else:
                    raise ValueError("Cannot find depth data in .mat file")
            
            # Determine the correct axis order
            # UTD-MHAD depth format is typically (320, 240, num_frames)
            if depth_data.shape[2] < depth_data.shape[0] and depth_data.shape[2] < depth_data.shape[1]:
                # Format: (height, width, frames) -> need to transpose
                num_frames_orig = depth_data.shape[2]
                frames = []
                for i in range(num_frames_orig):
                    frame = depth_data[:, :, i]
                    # Resize
                    frame = cv2.resize(frame, target_size)
                    # Normalize
                    frame = (frame - np.min(frame)) / (np.max(frame) - np.min(frame) + 1e-6)
                    frames.append(frame)
                frames = np.array(frames)
            else:
                # Format might be (frames, height, width)
                num_frames_orig = depth_data.shape[0]
                frames = []
                for i in range(num_frames_orig):
                    frame = depth_data[i, :, :]
                    frame = cv2.resize(frame, target_size)
                    frame = (frame - np.min(frame)) / (np.max(frame) - np.min(frame) + 1e-6)
                    frames.append(frame)
                frames = np.array(frames)
            
            # Resample to target frames
            if frames.shape[0] != target_frames:
                indices = np.linspace(0, frames.shape[0] - 1, target_frames).astype(int)
                frames = frames[indices]
            
            return frames.astype(np.float32)
        except Exception as e:
            print(f"Error loading depth {file_path}: {e}")
            import traceback
            traceback.print_exc()
            return np.zeros((target_frames, target_size[0], target_size[1]), dtype=np.float32)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        label = self.labels[idx]
        
        modality_data = {}
        
        if self.use_skeleton:
            skeleton = self.load_skeleton_data(sample['skeleton_path'])
            modality_data['skeleton'] = torch.FloatTensor(skeleton)
        
        if self.use_inertial:
            inertial = self.load_inertial_data(sample['inertial_path'])
            modality_data['inertial'] = torch.FloatTensor(inertial)
        
        if self.use_depth:
            depth = self.load_depth_data(sample['depth_path'])
            modality_data['depth'] = torch.FloatTensor(depth)
        
        return modality_data, torch.LongTensor([label])[0]

In [3]:
class SkeletonEncoder(nn.Module):
    """Encoder for skeleton data using LSTM"""
    def __init__(self, input_size=60, hidden_size=128, num_layers=2, dropout=0.3):
        super(SkeletonEncoder, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        self.bn = nn.BatchNorm1d(hidden_size * 2)
        self.fc = nn.Linear(hidden_size * 2, 256)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, frames, joints, coords)
        batch_size = x.size(0)
        x = x.view(batch_size, x.size(1), -1)
        
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        
        last_out = self.bn(last_out)
        features = self.dropout(torch.relu(self.fc(last_out)))
        return features

In [4]:
class InertialEncoder(nn.Module):
    """Encoder for inertial data using 1D CNN + LSTM"""
    def __init__(self, input_channels=6, hidden_size=128, dropout=0.3):
        super(InertialEncoder, self).__init__()
        
        # 1D CNN for inertial signals
        self.conv1 = nn.Conv1d(input_channels, 64, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(128)
        self.pool = nn.MaxPool1d(2)
        
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.bn3 = nn.BatchNorm1d(hidden_size * 2)
        self.fc = nn.Linear(hidden_size * 2, 256)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, frames, channels)
        x = x.permute(0, 2, 1)  # (batch, channels, frames)
        
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        
        x = x.permute(0, 2, 1)  # (batch, frames, channels)
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        
        last_out = self.bn3(last_out)
        features = self.dropout(torch.relu(self.fc(last_out)))
        return features

In [5]:
class DepthEncoder(nn.Module):
    """Encoder for depth video using 3D CNN"""
    def __init__(self, dropout=0.3):
        super(DepthEncoder, self).__init__()
        
        # 3D CNN for spatiotemporal features
        self.conv1 = nn.Conv3d(1, 32, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv2 = nn.Conv3d(32, 64, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.conv3 = nn.Conv3d(64, 128, kernel_size=(3, 3, 3), padding=(1, 1, 1))
        self.pool = nn.MaxPool3d((2, 2, 2))
        
        # Calculate size after convolutions
        # Input: (batch, 1, 50, 64, 64)
        # After 3 pools: (batch, 128, 6, 8, 8)
        self.fc = nn.Linear(128 * 6 * 8 * 8, 256)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch, frames, height, width)
        x = x.unsqueeze(1)  # (batch, 1, frames, height, width)
        
        x = torch.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = torch.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # Flatten
        features = self.dropout(torch.relu(self.fc(x)))
        return features

In [6]:
# ================== Multi-Modal Fusion Model ==================
class MultiModalFusionModel(nn.Module):
    """Multi-modal fusion model with late fusion strategy"""
    def __init__(self, num_classes=27, use_skeleton=True, use_inertial=True, 
                 use_depth=True, fusion_type='concat', dropout=0.5):
        super(MultiModalFusionModel, self).__init__()
        
        self.use_skeleton = use_skeleton
        self.use_inertial = use_inertial
        self.use_depth = use_depth
        self.fusion_type = fusion_type
        
        # Initialize encoders
        if use_skeleton:
            self.skeleton_encoder = SkeletonEncoder(dropout=dropout)
        
        if use_inertial:
            self.inertial_encoder = InertialEncoder(dropout=dropout)
        
        if use_depth:
            self.depth_encoder = DepthEncoder(dropout=dropout)
        
        # Fusion layer
        num_modalities = sum([use_skeleton, use_inertial, use_depth])
        
        if fusion_type == 'concat':
            fusion_input_size = 256 * num_modalities
        elif fusion_type == 'attention':
            fusion_input_size = 256
            self.attention_weights = nn.Linear(256 * num_modalities, num_modalities)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(fusion_input_size, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, modality_data):
        features = []
        
        if self.use_skeleton:
            skeleton_feat = self.skeleton_encoder(modality_data['skeleton'])
            features.append(skeleton_feat)
        
        if self.use_inertial:
            inertial_feat = self.inertial_encoder(modality_data['inertial'])
            features.append(inertial_feat)
        
        if self.use_depth:
            depth_feat = self.depth_encoder(modality_data['depth'])
            features.append(depth_feat)
        
        # Fusion
        if self.fusion_type == 'concat':
            fused = torch.cat(features, dim=1)
        elif self.fusion_type == 'attention':
            stacked_features = torch.stack(features, dim=1)  # (batch, num_modalities, 256)
            concat_features = torch.cat(features, dim=1)
            
            attention = torch.softmax(self.attention_weights(concat_features), dim=1)
            attention = attention.unsqueeze(-1)  # (batch, num_modalities, 1)
            
            fused = (stacked_features * attention).sum(dim=1)  # (batch, 256)
        
        # Classification
        output = self.classifier(fused)
        return output

In [7]:
# ================== Training Functions ==================
def train_epoch(model, dataloader, criterion, optimizer, device, scaler=None):
    """Train for one epoch with mixed precision support"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, labels) in enumerate(dataloader):
        # Move data to device
        data = {k: v.to(device, non_blocking=True) for k, v in data.items()}
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                outputs = model(data)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        if (batch_idx + 1) % 5 == 0:
            print(f"  Batch {batch_idx + 1}/{len(dataloader)}, Loss: {loss.item():.4f}, Acc: {100*correct/total:.2f}%")
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    
    return epoch_loss, epoch_acc


def evaluate(model, dataloader, criterion, device):
    """Evaluate the model with mixed precision"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for data, labels in dataloader:
            data = {k: v.to(device, non_blocking=True) for k, v in data.items()}
            labels = labels.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda'):
                outputs = model(data)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * accuracy_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, all_preds, all_labels

In [8]:
# ================== Main Training Script ==================
def main():
    # Configuration
    DATA_DIR = '/kaggle/input/utd-mhad-1'  # Root directory containing Skeleton/, Inertial/, Depth/
    BATCH_SIZE = 64  # Much larger batch size for dual T4 GPUs (32GB total)
    NUM_EPOCHS = 100
    LEARNING_RATE = 0.002  # Scaled learning rate for larger batch
    NUM_CLASSES = 27
    NUM_WORKERS = 4  # Parallel data loading
    
    # Multi-GPU setup
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_MULTI_GPU = torch.cuda.device_count() > 1
    
    print(f"Available GPUs: {torch.cuda.device_count()}")
    if USE_MULTI_GPU:
        print(f"Using {torch.cuda.device_count()} GPUs for training!")
    print(f"Primary device: {DEVICE}")
    
    # Modality selection
    USE_SKELETON = True
    USE_INERTIAL = True
    USE_DEPTH = True
    FUSION_TYPE = 'concat'  # 'concat' or 'attention'
    
    print(f"Using device: {DEVICE}")
    print(f"Multi-modal training with: Skeleton={USE_SKELETON}, Inertial={USE_INERTIAL}, Depth={USE_DEPTH}")
    print(f"Fusion strategy: {FUSION_TYPE}")
    
    # Create datasets
    print("\nLoading datasets...")
    train_dataset = UTDMHADMultiModalDataset(
        DATA_DIR, split='train', test_size=0.2,
        use_skeleton=USE_SKELETON, use_inertial=USE_INERTIAL, use_depth=USE_DEPTH
    )
    test_dataset = UTDMHADMultiModalDataset(
        DATA_DIR, split='test', test_size=0.2,
        use_skeleton=USE_SKELETON, use_inertial=USE_INERTIAL, use_depth=USE_DEPTH
    )
    
    # Create dataloaders with optimized settings
    train_loader = DataLoader(
        train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )
    test_loader = DataLoader(
        test_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True if NUM_WORKERS > 0 else False,
        prefetch_factor=2 if NUM_WORKERS > 0 else None
    )
    
    # Initialize model
    print("\nInitializing multi-modal fusion model...")
    model = MultiModalFusionModel(
        num_classes=NUM_CLASSES,
        use_skeleton=USE_SKELETON,
        use_inertial=USE_INERTIAL,
        use_depth=USE_DEPTH,
        fusion_type=FUSION_TYPE,
        dropout=0.5
    )
    
    # Multi-GPU parallelization
    if USE_MULTI_GPU:
        print(f"Wrapping model with DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    model = model.to(DEVICE)
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Loss and optimizer with gradient scaling for mixed precision
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    
    # Mixed precision training for faster computation
    scaler = torch.amp.GradScaler('cuda')
    
    # Cosine annealing with warmup
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-6
    )
    
    # Training loop
    print("\n" + "="*70)
    print("Starting multi-modal training...")
    print("="*70)
    best_acc = 0.0
    patience_counter = 0
    early_stop_patience = 10
    
    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
        print("-" * 70)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE, scaler)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        
        # Evaluate
        test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
        
        # Learning rate scheduling
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Learning Rate: {current_lr:.6f}")
        
        # Save best model
        if test_acc > best_acc:
            best_acc = test_acc
            patience_counter = 0
            
            # Save model state (handle DataParallel wrapper)
            model_to_save = model.module if USE_MULTI_GPU else model
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_to_save.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
                'test_acc': test_acc,
                'train_acc': train_acc,
                'config': {
                    'use_skeleton': USE_SKELETON,
                    'use_inertial': USE_INERTIAL,
                    'use_depth': USE_DEPTH,
                    'fusion_type': FUSION_TYPE
                }
            }, 'best_multimodal_model.pth')
            print(f"✓ Saved best model with accuracy: {best_acc:.2f}%")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= early_stop_patience:
            print(f"\nEarly stopping triggered after {epoch + 1} epochs")
            break
    
    # Final evaluation
    print("\n" + "="*70)
    print("Training completed!")
    print(f"Best Test Accuracy: {best_acc:.2f}%")
    print("="*70)

    # Load best model and detailed evaluation
    checkpoint = torch.load('best_multimodal_model.pth', weights_only=False)
    
    # Handle DataParallel wrapper
    if USE_MULTI_GPU:
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])
    
    _, _, final_preds, final_labels = evaluate(model, test_loader, criterion, DEVICE)
    
    # Action names for UTD-MHAD
    action_names = [
        "Swipe Left", "Swipe Right", "Wave", "Clap", "Throw", "Arm Cross",
        "Basketball Shoot", "Draw X", "Draw Circle CW", "Draw Circle CCW",
        "Draw Triangle", "Bowling", "Boxing", "Baseball Swing", "Tennis Swing",
        "Arm Curl", "Tennis Serve", "Push", "Knock", "Catch",
        "Pickup & Throw", "Jog", "Walk", "Sit to Stand", "Stand to Sit",
        "Lunge", "Squat"
    ]
    
    print("\nDetailed Classification Report:")
    print(classification_report(final_labels, final_preds, target_names=action_names, digits=3))
    
    print("\nPer-class Accuracy:")
    cm = confusion_matrix(final_labels, final_preds)
    class_acc = cm.diagonal() / cm.sum(axis=1)
    for i, (name, acc) in enumerate(zip(action_names, class_acc)):
        print(f"  {i+1:2d}. {name:20s}: {acc*100:5.2f}%")
    
    print(f"\nOverall Accuracy: {best_acc:.2f}%")
    print(f"Model saved to: best_multimodal_model.pth")


if __name__ == '__main__':
    main()

Available GPUs: 2
Using 2 GPUs for training!
Primary device: cuda
Using device: cuda
Multi-modal training with: Skeleton=True, Inertial=True, Depth=True
Fusion strategy: concat

Loading datasets...
TRAIN set: 688 samples
Modalities: Skeleton=True, Inertial=True, Depth=True
TEST set: 173 samples
Modalities: Skeleton=True, Inertial=True, Depth=True

Initializing multi-modal fusion model...
Wrapping model with DataParallel across 2 GPUs
Total parameters: 14,422,811
Trainable parameters: 14,422,811

Starting multi-modal training...

Epoch 1/100
----------------------------------------------------------------------
  Batch 5/11, Loss: 3.2351, Acc: 6.25%
  Batch 10/11, Loss: 3.1409, Acc: 8.91%
Train Loss: 3.2409, Train Acc: 8.87%
Test Loss: 3.2319, Test Acc: 23.12%
Learning Rate: 0.001951
✓ Saved best model with accuracy: 23.12%

Epoch 2/100
----------------------------------------------------------------------
  Batch 5/11, Loss: 2.8317, Acc: 17.19%
  Batch 10/11, Loss: 2.6665, Acc: 17.97%
